# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaimAli0001/Flyrank-Internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method: Logistic Regression

This project is a Refresh / Content Opportunity Scoring task where the goal is to rank pages for review.

Logistic Regression was selected as the first learned model because it is simple, interpretable, and produces a probability score that can be used to rank pages. This fits the "which first?" nature of the task and provides a clear first learned model to compare against the frozen Week-4 rule baseline.

The target is `ctr_decline`, defined as whether a page's April 2026 CTR is lower than its March 2026 CTR. March search-performance features are used as inputs and April CTR decline is treated as the future observed outcome.

This target does not claim that a page should definitely be refreshed. It measures a future performance change and is used as decision-support evidence for ranking review candidates.

A more complex model will only be considered if it provides a meaningful improvement over this interpretable baseline.

In [47]:
from dotenv import load_dotenv
import os
import duckdb

load_dotenv("../../.env")

token = os.getenv("HF_TOKEN")
print("Token loaded:", token is not None)

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

feature_vector = con.sql(f"""
WITH daily_metrics AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {REL}
    WHERE gsc_data_available IS TRUE
),

half_month_metrics AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position,

        AVG(
            CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN gsc_impressions
            END
        ) AS first_half_avg_impressions,

        AVG(
            CASE
                WHEN report_date > DATE '2026-03-15'
                THEN gsc_impressions
            END
        ) AS second_half_avg_impressions,

        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS first_half_clicks,

        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS first_half_impressions,

        SUM(
            CASE
                WHEN report_date > DATE '2026-03-15'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS second_half_clicks,

        SUM(
            CASE
                WHEN report_date > DATE '2026-03-15'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS second_half_impressions

    FROM daily_metrics

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,

    march_impressions,

    CASE
        WHEN march_impressions > 0
        THEN march_clicks * 1.0 / march_impressions
        ELSE NULL
    END AS march_ctr,

    march_avg_position,

    second_half_avg_impressions
        - first_half_avg_impressions
        AS impression_trend,

    CASE
        WHEN second_half_impressions > 0
         AND first_half_impressions > 0
        THEN
            (second_half_clicks * 1.0 / second_half_impressions)
            -
            (first_half_clicks * 1.0 / first_half_impressions)
        ELSE NULL
    END AS ctr_trend

FROM half_month_metrics
""").df()

feature_vector.shape

Token loaded: True


(176738, 7)

In [48]:
feature_vector.head()

,client_hash_id,content_hash_id,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend
0,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,0.003185,9.155335,-2.591667,0.007042
1,client_73cda7b4e4f265ea,content_6cf70a5cb30e76f2,3465.0,0.001443,7.748876,-8.187500,-0.001719
2,client_73cda7b4e4f265ea,content_0f040e1b2668c08a,74.0,0.000000,60.670679,-1.390110,0.000000
3,client_73cda7b4e4f265ea,content_0d510f7a6abb761e,1431.0,0.003494,5.833380,5.608333,-0.004885
4,client_73cda7b4e4f265ea,content_c58edf8de191403f,16.0,0.000000,12.854167,-0.666667,0.000000


In [49]:
april_outcome_check = con.sql("""
WITH april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    COUNT(*) AS april_pages,
    COUNT(*) FILTER (
        WHERE april_impressions > 0
    ) AS pages_with_impressions,
    COUNT(*) FILTER (
        WHERE april_clicks > 0
    ) AS pages_with_clicks,
    AVG(april_impressions) AS avg_april_impressions,
    MEDIAN(april_impressions) AS median_april_impressions
FROM april
JOIN feature_vector f
    USING (client_hash_id, content_hash_id)
""").df()

april_outcome_check

,april_pages,pages_with_impressions,pages_with_clicks,avg_april_impressions,median_april_impressions
0,158549,158549,59221,1777.749194,200.0


In [50]:
april_distribution = con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        march_impressions,
        march_ctr
    FROM feature_vector
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS april_ctr
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    COUNT(*) AS n,
    AVG(april_ctr) AS avg_april_ctr,
    MEDIAN(april_ctr) AS median_april_ctr,
    AVG(
        CASE
            WHEN march_ctr > 0 AND april_ctr IS NOT NULL
            THEN april_ctr - march_ctr
        END
    ) AS avg_ctr_change
FROM march
JOIN april
    USING (client_hash_id, content_hash_id)
""").df()

april_distribution

,n,avg_april_ctr,median_april_ctr,avg_ctr_change
0,158549,0.003028,0.0,-0.005217


In [51]:
ctr_change_distribution = con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        march_ctr
    FROM feature_vector
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS april_ctr
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    MIN(april_ctr - march_ctr) AS minimum_change,
    quantile_cont(april_ctr - march_ctr, 0.10) AS p10,
    quantile_cont(april_ctr - march_ctr, 0.25) AS q1,
    MEDIAN(april_ctr - march_ctr) AS median_change,
    quantile_cont(april_ctr - march_ctr, 0.75) AS q3,
    quantile_cont(april_ctr - march_ctr, 0.90) AS p90,
    MAX(april_ctr - march_ctr) AS maximum_change
FROM march
JOIN april
    USING (client_hash_id, content_hash_id)
WHERE april_ctr IS NOT NULL
""").df()

ctr_change_distribution

,minimum_change,p10,q1,median_change,q3,p90,maximum_change
0,-1.0,-0.003387,-0.000572,0.0,0.0,0.001707,1.0


In [52]:
decline_counts = con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        march_ctr
    FROM feature_vector
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS april_ctr
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

outcomes AS (
    SELECT
        march.client_hash_id,
        march.content_hash_id,
        april.april_ctr - march.march_ctr AS ctr_change
    FROM march
    JOIN april
        USING (client_hash_id, content_hash_id)
    WHERE april.april_ctr IS NOT NULL
)

SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE ctr_change < 0) AS declined,
    COUNT(*) FILTER (WHERE ctr_change >= 0) AS not_declined,
    AVG(CASE WHEN ctr_change < 0 THEN 1.0 ELSE 0.0 END) AS decline_rate
FROM outcomes
""").df()

decline_counts

,total,declined,not_declined,decline_rate
0,158549,48756,109793,0.307514


In [53]:
april_outcomes = con.sql("""
WITH april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.march_ctr,
    a.april_impressions,
    CASE
        WHEN a.april_impressions > 0
        THEN a.april_clicks * 1.0 / a.april_impressions
        ELSE NULL
    END AS april_ctr
FROM feature_vector f
JOIN april a
    USING (client_hash_id, content_hash_id)
""").df()

april_outcomes["ctr_decline"] = (
    april_outcomes["april_ctr"] < april_outcomes["march_ctr"]
).astype(int)

model_data = (
    feature_vector
    .merge(
        april_outcomes[
            ["client_hash_id", "content_hash_id", "ctr_decline"]
        ],
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)

print("Model rows:", len(model_data))
print("Positive rate:", model_data["ctr_decline"].mean())

Model rows: 158549
Positive rate: 0.3075137654605201


In [54]:
FEATURES = [
    "march_impressions",
    "march_ctr",
    "march_avg_position",
    "impression_trend",
    "ctr_trend",
]

TARGET = "ctr_decline"

print(model_data[FEATURES + [TARGET]].isna().sum())

march_impressions         0
march_ctr                 0
march_avg_position        0
impression_trend      23291
ctr_trend             23291
ctr_decline               0
dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

A client-grouped holdout is used so that the model is evaluated on clients that were completely unseen during training.

This is more honest than a random row split because pages from the same client can share traffic and performance patterns. The `client_hash_id` is used only for grouping and is not used as a predictive feature.

An 80/20 client-grouped split is used with a fixed random seed of 42 for reproducibility.

In [55]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

X = model_data[FEATURES].copy()
y = model_data[TARGET].copy()
groups = model_data["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_groups = groups.iloc[train_idx]
test_groups = groups.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", train_groups.nunique())
print("Test clients:", test_groups.nunique())
print("Client overlap:", len(set(train_groups) & set(test_groups)))

Train rows: 137445
Test rows: 21104
Train clients: 36
Test clients: 10
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Logistic Regression model is trained using the client-grouped training set from Section 2.

The model produces a probability of future CTR decline for each test page. Pages are ranked by this probability, and the top 20 are treated as the highest-priority review candidates.

The Week-4 transparent baseline is kept frozen. Its original scoring rules are applied to the same test pages without changing any thresholds after seeing the model results.

Both systems are compared using Precision@20 on the same test pages and the same future outcome label (`ctr_decline`).

In [56]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
K = 20

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000
    ))
])

logistic_model.fit(X_train, y_train)

model_prob = logistic_model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")
print("Test predictions:", len(model_prob))

Model trained successfully.
Test predictions: 21104


In [57]:
import pandas as pd

In [58]:
# Create the test-page result table
model_results = model_data.iloc[test_idx][
    [
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "march_ctr",
        "march_avg_position",
        "impression_trend",
        "ctr_trend",
        "ctr_decline",
    ]
].copy()

model_results["model_probability"] = model_prob


# -----------------------------
# Frozen W04 baseline
# -----------------------------

def impression_score(x):
    if x <= 20:
        return 1
    elif x <= 173:
        return 2
    elif x <= 1039:
        return 3
    elif x <= 5000:
        return 4
    else:
        return 5


def ctr_score(x):
    if x == 0:
        return 5
    elif x <= 0.002:
        return 4
    elif x <= 0.005:
        return 3
    elif x <= 0.01:
        return 2
    else:
        return 1


def position_score(x):
    if 4 <= x <= 7:
        return 5
    elif 8 <= x <= 12:
        return 4
    elif 13 <= x <= 20:
        return 3
    elif x >= 21:
        return 2
    else:
        return 1


model_results["impression_score"] = (
    model_results["march_impressions"].apply(impression_score)
)

model_results["ctr_score"] = (
    model_results["march_ctr"].apply(ctr_score)
)

model_results["position_score"] = (
    model_results["march_avg_position"].apply(position_score)
)

model_results["priority_score"] = (
    model_results["impression_score"]
    + model_results["ctr_score"]
    + model_results["position_score"]
)



K = 20

def precision_at_k(df, score_column, k=20):
    ranked = df.sort_values(
        by=[score_column, "march_impressions", "march_ctr"],
        ascending=[False, False, True]
    )
    
    return ranked.head(k)["ctr_decline"].mean()


model_precision_20 = precision_at_k(
    model_results,
    "model_probability",
    K
)

baseline_precision_20 = precision_at_k(
    model_results,
    "priority_score",
    K
)

base_rate = model_results["ctr_decline"].mean()


comparison = pd.DataFrame({
    "method": [
        "W04 baseline",
        "Logistic Regression"
    ],
    "precision_at_20": [
        baseline_precision_20,
        model_precision_20
    ],
    "base_rate": [
        base_rate,
        base_rate
    ]
})

comparison

,method,precision_at_20,base_rate
0,W04 baseline,0.8,0.420015
1,Logistic Regression,1.0,0.420015


In [59]:
top20_model = (
    model_results
    .sort_values(
        by=["model_probability", "march_impressions", "march_ctr"],
        ascending=[False, False, True]
    )
    .head(20)
)

top20_model[
    [
        "march_impressions",
        "march_ctr",
        "march_avg_position",
        "impression_trend",
        "ctr_trend",
        "model_probability",
        "ctr_decline"
    ]
]

,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend,model_probability,ctr_decline
155407,35.0,0.142857,5.074074,-0.777778,0.000000,1.0,1
7038,17.0,0.117647,4.766667,0.777778,0.125000,1.0,1
27291,16.0,0.125000,4.088889,NaN,NaN,1.0,1
61702,13.0,0.076923,44.000000,0.466667,-0.250000,1.0,1
57307,12.0,0.166667,6.916667,-0.500000,-0.285714,1.0,1
9703,11.0,0.090909,4.156250,7.000000,-0.333333,1.0,1
68317,11.0,0.090909,25.833333,-0.357143,-0.333333,1.0,1
36754,10.0,0.100000,14.333333,-0.250000,-0.200000,1.0,1
63169,9.0,0.111111,3.850000,-0.333333,-0.142857,1.0,1
83973,9.0,0.111111,24.166667,-0.750000,-0.142857,1.0,1


In [60]:
top20_baseline = (
    model_results
    .sort_values(
        by=["priority_score", "march_impressions", "march_ctr"],
        ascending=[False, False, True]
    )
    .head(20)
)

top20_baseline[
    [
        "march_impressions",
        "march_ctr",
        "march_avg_position",
        "priority_score",
        "ctr_decline"
    ]
]

,march_impressions,march_ctr,march_avg_position,priority_score,ctr_decline
84981,49994.0,0.001960,4.623840,14,1
23128,48309.0,0.000600,6.813802,14,1
94490,38188.0,0.001571,4.028375,14,1
57311,34721.0,0.001267,6.996807,14,1
50065,30288.0,0.001453,6.189506,14,1
73328,28795.0,0.001285,4.952878,14,1
117111,22906.0,0.001135,6.407122,14,1
25670,21603.0,0.000139,5.891773,14,0
47923,20220.0,0.001830,6.066966,14,1
134154,20020.0,0.000350,4.025142,14,0


In [61]:
def precision_at_k(df, score_column, k=20):
    ranked = df.sort_values(
        by=[score_column, "march_impressions", "march_ctr"],
        ascending=[False, False, True]
    )
    return ranked.head(k)["ctr_decline"].mean()


k_values = [20, 50, 100]

comparison_multi_k = pd.DataFrame({
    "K": k_values,
    "W04_baseline": [
        precision_at_k(model_results, "priority_score", k)
        for k in k_values
    ],
    "Logistic_Regression": [
        precision_at_k(model_results, "model_probability", k)
        for k in k_values
    ]
})

comparison_multi_k

,K,W04_baseline,Logistic_Regression
0,20,0.80,1.00
1,50,0.72,1.00
2,100,0.56,0.99


### Result

The Logistic Regression model achieved higher observed Precision@K than the frozen Week-4 baseline on the held-out client-grouped test split at all tested review depths.

- At Precision@20, Logistic Regression achieved 1.00 compared with 0.80 for the Week-4 baseline.
- At Precision@50, Logistic Regression achieved 1.00 compared with 0.72.
- At Precision@100, Logistic Regression achieved 0.99 compared with 0.56.

The test-set base rate was approximately 0.42. These results show strong observed ranking performance for Logistic Regression on this split, but they do not establish that the model will generalize equally well to other clients or future periods.

In [62]:
comparison_final = comparison_multi_k.copy()

comparison_final["base_rate"] = model_results["ctr_decline"].mean()

comparison_final

,K,W04_baseline,Logistic_Regression,base_rate
0,20,0.80,1.00,0.420015
1,50,0.72,1.00,0.420015
2,100,0.56,0.99,0.420015


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Logistic Regression model relies most strongly on March CTR, followed by CTR trend and March impressions. These relationships are directionally plausible because current click-through behaviour and recent CTR movement can be associated with subsequent performance changes.

The model made 4,865 errors out of 21,104 test pages, giving an overall error rate of approximately 23.05%, despite achieving very high Precision@K at the top of the ranking.

The top-ranked pages were heavily concentrated among very low-impression pages. The median impression count among the model's Top-20 was only 8, meaning that small changes in clicks can create large changes in CTR. This makes the future decline label less stable for low-volume pages.

Three high-confidence false positives were observed where the model assigned probabilities near 1.0 but the April CTR did not decline. These cases show that the model can be highly confident when the available March signals resemble pages that often experience decline, even when the future outcome does not follow that pattern.

Overall, the model shows strong observed ranking performance on this client-grouped test split, but the results should be treated as decision-support evidence rather than proof of general superiority or guaranteed future performance.

In [63]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

coefficients = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": logistic_model.named_steps["model"].coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

coefficients

,feature,coefficient,absolute_coefficient
1,march_ctr,10.305895,10.305895
4,ctr_trend,-1.315466,1.315466
0,march_impressions,0.552207,0.552207
2,march_avg_position,-0.406850,0.406850
3,impression_trend,0.068224,0.068224


In [64]:
model_results["predicted_class"] = (
    model_results["model_probability"] >= 0.5
).astype(int)

errors = model_results[
    model_results["predicted_class"] != model_results["ctr_decline"]
].copy()

print("Total test rows:", len(model_results))
print("Total errors:", len(errors))
print("Error rate:", len(errors) / len(model_results))

Total test rows: 21104
Total errors: 4865
Error rate: 0.23052501895375285


In [65]:
error_examples = (
    errors
    .sort_values(
        by="model_probability",
        ascending=False
    )
    .head(3)
)

error_examples[
    [
        "march_impressions",
        "march_ctr",
        "march_avg_position",
        "impression_trend",
        "ctr_trend",
        "model_probability",
        "ctr_decline"
    ]
]

,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend,model_probability,ctr_decline
89477,108.0,0.046296,7.158621,-0.157143,-0.013416,1.000000,0
91077,91.0,0.032967,5.687252,-4.583333,-0.036585,0.999996,0
136889,29.0,0.034483,4.306944,NaN,NaN,0.999990,0


In [66]:
top20_volume_check = (
    model_results
    .sort_values(
        "model_probability",
        ascending=False
    )
    .head(20)
)

top20_volume_check["march_impressions"].describe()

count    20.000000
mean      9.400000
std       7.081481
min       2.000000
25%       5.000000
50%       8.500000
75%      11.000000
max      35.000000
Name: march_impressions, dtype: float64

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.